In [1]:
import os
usda_api_key = os.environ["MY_USDA_API_KEY"]
from usda_fdc import FdcClient
import pickle
from utils import usda_codes
from tqdm import tqdm
import numpy as np
from utils import fda_list, nasem_list

with open('data/burgers_step8.pkl', 'rb') as f:
    recipes, ingr_names, calorie_database = pickle.load(f)
n_ingr = len(ingr_names)

In [2]:
print(len(fda_list), len(nasem_list))

35 31


In [5]:
# add fatty acid totals to the list
nasem_list['polyunsaturated'] = [None, None, None]
nasem_list['monounsaturated'] = [None, None, None]
nasem_list[' saturated'] = [None, None, None]
nasem_list['fat'] = [None, None, None]

# add total sugars to the list
nasem_list['total sugars'] = [None, None, None]

In [10]:
def handle_exceptions(nutrient, matches_in_ingr):
    # first of all, if there is an exact match, prefer that
    for match in matches_in_ingr:
        if match == nutrient:
            return [match]
    
    if nutrient == 'fat':
        for match in matches_in_ingr:
            if 'total lipid' in match:
                return [match]
            elif 'total fat' in match:
                return [match]
        else:
            return matches_in_ingr
    
    if nutrient == 'vitamin a':
        for match in matches_in_ingr:
            if 'rae' in match:
                return [match]
        else:
            return matches_in_ingr
        
    if nutrient == 'vitamin d':
        for match in matches_in_ingr:
            if 'vitamin d (d2 + d3)' in match:
                return [match]
        else:
            return matches_in_ingr
    
    if nutrient == 'vitamin e':
        for match in matches_in_ingr:
            if 'vitamin e (alphatocopherol)' in match:
                return [match]
        else:
            return matches_in_ingr
    
    if nutrient == 'vitamin k':
        for match in matches_in_ingr:
            if 'vitamin k (phylloquinone)' in match:
                return [match]
        else:
            return matches_in_ingr

    if nutrient == 'folate':
        for match in matches_in_ingr:
            if 'folate, total' in match:
                return [match]
        else:
            return matches_in_ingr

    if nutrient == 'carbohydrate':
        for match in matches_in_ingr:
            if 'carbohydrates' in match:
                return [match]
        else:
            return matches_in_ingr

    if nutrient == 'saturated':
        for match in matches_in_ingr:
            match_split = match.split(" ")
            if 'saturated' in match_split:
                return [match]
        else:
            return matches_in_ingr

    if nutrient == 'pufa 18:2':
        for match in matches_in_ingr:
            if 'pufa 18:2 n6' in match:
                return [match]
        else:
            return matches_in_ingr
        
    if nutrient == 'pufa 18:3':
        for match in matches_in_ingr:
            if 'pufa 18:3 n3' in match:
                return [match]
        else:
            return matches_in_ingr
    
    if nutrient == 'polyunsaturated':
        for match in matches_in_ingr:
            if 'total' in match:
                return [match]
        else:
            return matches_in_ingr
    
    if nutrient == 'monounsaturated':
        for match in matches_in_ingr:
            if 'total' in match:
                return [match]
        else:
            return matches_in_ingr
    
    if nutrient == 'saturated':
        for match in matches_in_ingr:
            if 'total' in match:
                return [match]
        else:
            return matches_in_ingr
        
        
    return matches_in_ingr


In [11]:
ingredient_nutrient_amounts = np.zeros([n_ingr, len(nasem_list)])
ingredient_nutrient_units = np.empty([n_ingr, len(nasem_list)], dtype='U10') 

client = FdcClient(usda_api_key)
for i, ingr in tqdm(enumerate(ingr_names), total = n_ingr):
    food_id = usda_codes[ingr]
    food = client.get_food(food_id)

    # get serving size for normalization
    if food.serving_size == None:
        serving_size = 100
    elif food.serving_size == 'g' or food.serving_size == 'GRM':
        serving_size = 1
    elif food.serving_size_unit == 'g' or food.serving_size_unit == 'GRM':
        serving_size = food.serving_size

    ingr_nutrient_list = client.get_nutrients(food_id)
    for j, nutrient in enumerate(nasem_list):
        matches_in_ingr = []
        for ingr_nutrient in ingr_nutrient_list:
            ingr_nutrient_name = ingr_nutrient.name.lower().replace("-", "")
            if nutrient in ingr_nutrient_name:
                matches_in_ingr.append(ingr_nutrient_name)
        
        if len(matches_in_ingr) > 1:
            matches_in_ingr = handle_exceptions(nutrient, matches_in_ingr)
        
        if len(matches_in_ingr) > 1:
            print('-'*30)
            print(nutrient)
            print(matches_in_ingr)
        
        if len(matches_in_ingr) > 0:
            match = matches_in_ingr[0]
            match_all_info = next((sub for sub in ingr_nutrient_list if sub.name.lower().replace("-", "") == match), None) # this just goes back to ingr_nutrient_list to find qnty, units, etc
            ingredient_nutrient_amounts[i, j] = match_all_info.amount/serving_size
            ingredient_nutrient_units[i, j] = match_all_info.unit_name


100%|██████████| 146/146 [01:26<00:00,  1.69it/s]


In [12]:
# unify units, because some nutrients use multiple units
for i, nutrient in enumerate(nasem_list):
    for j in range(n_ingr):
        if ingredient_nutrient_units[j, i] == 'µg':
            ingredient_nutrient_units[j, i] = 'mcg'

for i, nutrient in enumerate(nasem_list):
    if nutrient == 'vitamin a':
        for j in range(n_ingr):
            unit = ingredient_nutrient_units[j, i]
            if unit == 'IU':
                ingredient_nutrient_units[j, i] = 'mcg' # Since we are using "retinol activity equivalents (rae)", IU and mcg are the same
            elif unit == 'mg':
                ingredient_nutrient_amounts[j, i] = ingredient_nutrient_amounts[j, i] * 1000
                ingredient_nutrient_units[j, i] = 'mcg'
    elif nutrient == 'vitamin d':
        for j in range(n_ingr):
            unit = ingredient_nutrient_units[j, i]
            if unit == 'IU':
                ingredient_nutrient_amounts[j, i] = ingredient_nutrient_amounts[j, i] * 0.025
                ingredient_nutrient_units[j, i] = 'mcg'
    
# When I checked, only vitamin a and vitamin d had more than one unit. But double check.
for i, nutrient in enumerate(nasem_list):
    sublist = ingredient_nutrient_units[:,i]
    non_empty_elements = [s for s in sublist if s]
    unique_units, counts = np.unique(non_empty_elements, return_counts=True)

    if len(unique_units) > 1:
        print('-'*30)
        print(nutrient)
        for j in range(len(counts)):
            print(unique_units[j], counts[j])

In [13]:
# convert EVERYTHING to grams
rdvs = []
for nutrient in nasem_list:
    amount, unit, _ = nasem_list[nutrient]
    if unit == 'mg':
        rdvs.append(amount/1000)
    elif unit == 'mcg':
        rdvs.append(amount/1000000)
    else:
        rdvs.append(amount)
rdvs = np.array(rdvs, dtype='float64')


ingredient_nutrients = np.zeros([n_ingr, len(nasem_list)], dtype='float64')
for i in range(n_ingr):
    for j in range(len(nasem_list)):
        if ingredient_nutrient_units[i,j] == 'mg':
            ingredient_nutrients[i,j] = ingredient_nutrient_amounts[i,j]/1000
        elif ingredient_nutrient_units[i,j] == 'mcg':
            ingredient_nutrients[i,j] = ingredient_nutrient_amounts[i,j]/1000000
        elif ingredient_nutrient_units[i,j] == 'g':
            ingredient_nutrients[i,j] = ingredient_nutrient_amounts[i,j]
        else:
            print(ingredient_nutrient_units[i,j])

In [14]:
# check to see if there are any nutrients that are not listed in ANY ingredient, as this would be a major red flag.
for i, nutrient in enumerate(nasem_list):
    out = max(ingredient_nutrient_amounts[:,i])
    if out == 0:
        print(nutrient)

In [15]:
with open('data/ingredient_nutrient_info.pkl', 'wb') as f:
    pickle.dump([list(nasem_list), rdvs, ingredient_nutrients], f) # nutrient names, recommended daily values, ingredient nutrients

In [18]:
import pandas as pd

In [30]:
nasem_data_male = pd.read_excel('data/nutritional/nasem_dri.xlsx', sheet_name='male')

In [32]:
nasem_data_male

,age_lower (y),age_upper (y),calcium (mg/d),chromium (microgr/d),copper (microgr/d),fluoride (mg/d),iodine (microgr/d),iron (mg/d),magnesium (mg/d),manganese (mg/d),...,pantothenic acid (mg/d),biotin (microgr/d),choline (mg/d),total water (L/d),carbohydrate (g/d),total fiber (g/d),fat (g/d),linoleic acid (g/d),alpha-linolenic acid (g/d),protein (g/d)
0,0.0,0.5,200,0.2,200,0.01,110,0.27,30,0.003,...,1.7*,5*,125*,0.7*,60*,ND,31*,4.4*,0.5*,9.1*
1,0.5,1.0,260,5.5,220,0.50,130,11.00,75,0.600,...,1.8*,6*,150*,0.8*,95*,ND,30*,4.6*,0.5*,11
2,1.0,3.0,700,11.0,340,0.70,90,7.00,80,1.200,...,2*,8*,200*,1.3*,130,19*,NDc,7*,0.7*,13
3,4.0,8.0,1000,15.0,440,1.00,90,10.00,130,1.500,...,3*,12*,250*,1.7*,130,25*,ND,10*,0.9*,19
4,9.0,13.0,1300,25.0,700,2.00,120,8.00,240,1.900,...,4*,20*,375*,2.4*,130,31*,ND,12*,1.2*,34
5,14.0,18.0,1300,35.0,890,3.00,150,11.00,410,2.200,...,5*,25*,550*,3.3*,130,38*,ND,16*,1.6*,52
6,19.0,30.0,1000,35.0,900,4.00,150,8.00,400,2.300,...,5*,30*,550*,3.7*,130,38*,ND,17*,1.6*,56
7,31.0,50.0,1000,35.0,900,4.00,150,8.00,420,2.300,...,5*,30*,550*,3.7*,130,38*,ND,17*,1.6*,56
8,51.0,70.0,1000,30.0,900,4.00,150,8.00,420,2.300,...,5*,30*,550*,3.7*,130,30*,ND,14*,1.6*,56
9,71.0,1000.0,1200,30.0,900,4.00,150,8.00,420,2.300,...,5*,30*,550*,3.7*,130,30*,ND,14*,1.6*,56


In [33]:
nasem_data_male.iloc[0]

age_lower (y)                   0.0
age_upper (y)                   0.5
calcium (mg/d)                  200
chromium (microgr/d)            0.2
copper (microgr/d)              200
fluoride (mg/d)                0.01
iodine (microgr/d)              110
iron (mg/d)                    0.27
magnesium (mg/d)                 30
manganese (mg/d)              0.003
molybdenum (microgr/d)            2
phosphorus (mg/d)               100
selenium (microgr/d)             15
zinc (mg/d)                       2
potassium (mg/d)                400
sodium (mg/d)                   110
chloride (g/d)                 0.18
vitamin a (microgr/d)          400*
vitamin c (mg/d)                40*
vitamin d (microgr/d)           10*
vitamin e (mg/d)                 4*
vitamin k (microgr/d)          2.0*
thiamin (mg/d)                 0.2*
riboflavin (mg/d)              0.3*
niacin (mg/d)                    2*
vitamin b6 (mg/d)              0.1*
folate (microgr/d)              65*
vitamin b12 (microgr/d)     

In [35]:
i_group = 1
nasem_group_data = nasem_data_male.iloc[i_group]
bc = nasem_group_data[r'chromium (microgr/d)']

In [36]:
bc

5.5

In [27]:

b, c = bc.split('-')
b, c = float(b), float(c)